# Sketch2Shape AI — Colab Evaluation

Run on **Google Colab → T4 GPU**. This notebook deliberately does not install `requirements.txt` and does not replace Colab's CUDA-enabled PyTorch.

## 1. Runtime check

In [ ]:
import os, sys, time, subprocess, urllib.request, shutil, importlib
import torch
print("Python:", sys.version)
print("Torch:", torch.__version__, "CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT AVAILABLE")
if not torch.cuda.is_available():
    raise RuntimeError("Enable T4 GPU in Runtime > Change runtime type.")

## 2. Clone repositories

In [ ]:
%cd /content
!git clone https://github.com/pnkj006/Sketch2Shape-AI.git
!git clone https://github.com/TencentARC/InstantMesh.git
%cd /content/Sketch2Shape-AI

## 3. Install isolated NumPy and SciPy

These go into a separate directory to avoid changing Colab's system-managed NumPy.

In [ ]:
%pip install -q --no-cache-dir --no-deps --target=/content/s2s_deps numpy==2.2.6
%pip install -q --no-cache-dir --no-deps --target=/content/s2s_deps scipy==1.15.3
import sys
sys.path.insert(0, "/content/s2s_deps")
import numpy as np, numpy.typing as npt, scipy
from scipy import sparse
print("NumPy", np.__version__, np.__file__)
print("SciPy", scipy.__version__, scipy.__file__)
print("numpy.typing OK:", npt.__file__)

## 4. Install pinned project dependencies

PyTorch is intentionally left as supplied by Colab. `--no-deps` prevents pip from replacing Torch, NumPy, or SciPy.

In [ ]:
%pip install -q --no-deps \
diffusers==0.42.1 transformers==5.21.0 accelerate==1.12.0 safetensors==0.7.0 \
huggingface_hub==0.36.2 gradio==6.30.0 fastapi==0.141.1 uvicorn==0.41.0 \
trimesh==5.2.0 PyMCubes==0.1.6 xatlas==0.0.11 einops==0.8.1 omegaconf==2.3.0 \
timm==1.0.24 opencv-python-headless==4.11.0.86 pillow==11.3.0 rembg==2.0.67 \
onnxruntime==1.23.2 imageio==2.37.0 imageio-ffmpeg==0.6.0 ninja==1.13.0
%pip install -q --no-deps protobuf sentencepiece peft

## 5. Verify dependencies and CUDA compiler

In [ ]:
import sys, os, shutil, importlib
sys.path.insert(0, "/content/s2s_deps")
import numpy as np, scipy
print("Torch:", torch.__version__, "CUDA:", torch.version.cuda)
print("NumPy/SciPy:", np.__version__, scipy.__version__)
nvcc = shutil.which("nvcc")
print("nvcc:", nvcc)
if not nvcc:
    raise RuntimeError("nvcc not found. Use a Colab GPU runtime with CUDA toolkit.")
os.environ["CUDA_HOME"] = os.path.dirname(os.path.dirname(nvcc))
for name in ["diffusers","transformers","accelerate","trimesh","mcubes","xatlas","gradio","fastapi"]:
    m = importlib.import_module(name)
    print(name, getattr(m, "__version__", "OK"))

## 6. Build and install nvdiffrast

In [ ]:
%cd /content
!git clone https://github.com/NVlabs/nvdiffrast.git
%cd /content/nvdiffrast
%pip install -q --no-build-isolation .
%cd /content/Sketch2Shape-AI
import nvdiffrast.torch
print("nvdiffrast OK")

## 7. Apply compatibility patch and import project

In [ ]:
import torch
import transformers.pytorch_utils as pytorch_utils

def find_pruneable_heads_and_indices(heads, n_heads, head_size, already_pruned_heads):
    mask = torch.ones(n_heads, head_size)
    heads = set(heads) - already_pruned_heads
    for head in heads:
        head -= sum(1 for h in already_pruned_heads if h < head)
        mask[head] = 0
    index = torch.arange(mask.numel())[mask.view(-1).bool()]
    return heads, index.long()

pytorch_utils.find_pruneable_heads_and_indices = find_pruneable_heads_and_indices
from src.models.encoder.dino import ViTModel

def compatible_get_head_mask(self, head_mask, num_hidden_layers):
    if head_mask is None:
        return [None] * num_hidden_layers
    return self._convert_head_mask_to_5d(head_mask, num_hidden_layers)

ViTModel.get_head_mask = compatible_get_head_mask

from ml.mesh_validation import MeshValidator
from ml.preprocessing import preprocess_sketch, remove_background
from ml.reconstruction import Reconstruction
from ml.image_generation import ImageGenerator
from services.model_loader import ModelLoader
from services.pipeline import Sketch2ShapePipeline
print("Project imports OK")

## 8. Check required files

In [ ]:
from pathlib import Path
required = ["backend/main.py","frontend/app.py","services/pipeline.py","services/model_loader.py",
            "ml/preprocessing.py","ml/image_generation.py","ml/reconstruction.py","ml/mesh_validation.py"]
missing = [x for x in required if not Path(x).exists()]
if missing: raise FileNotFoundError(missing)
Path("outputs").mkdir(exist_ok=True)
print("Project files OK")

## 9. Start API backend and check health

In [ ]:
backend_log = open("/content/s2s_backend.log","w")
backend = subprocess.Popen([sys.executable,"-m","uvicorn","backend.main:app","--host","0.0.0.0","--port","8000"],
                           stdout=backend_log, stderr=subprocess.STDOUT)
url = "http://127.0.0.1:8000/api/health"
for _ in range(60):
    try:
        with urllib.request.urlopen(url,timeout=3) as r:
            print("Backend:",r.status,r.read().decode()); break
    except Exception: time.sleep(2)
else:
    print(open("/content/s2s_backend.log").read()[-5000:])
    raise RuntimeError("Backend failed to start.")

## 10. Start Gradio and open the app

In [ ]:
frontend_log = open("/content/s2s_frontend.log","w")
frontend = subprocess.Popen([sys.executable,"frontend/app.py"], stdout=frontend_log, stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        with urllib.request.urlopen("http://127.0.0.1:7861",timeout=3) as r:
            print("Gradio:",r.status); break
    except Exception: time.sleep(2)
else:
    print(open("/content/s2s_frontend.log").read()[-5000:])
    raise RuntimeError("Gradio failed to start.")

In [ ]:
from google.colab import output
output.serve_kernel_port_as_window(7861)

## 11. Evaluation

- Draw a sketch or upload an image.
- Add an optional prompt and generate.
- Inspect the interactive 3D preview.
- Download the `.glb` or `.obj` mesh.
- Record the actual generation time observed in this runtime.

**For low GPU memory, keep InstantMesh `grid_res=64` in the project configuration.**

## Troubleshooting

- If NumPy/SciPy fails, restart the runtime and rerun from Section 1; do not install `requirements.txt`.
- If `nvcc` or `nvdiffrast` fails, capture the complete build error; the CUDA toolkit may not match this runtime.
- If GPU memory runs out, verify `grid_res=64`.
- Backend and frontend logs: `/content/s2s_backend.log` and `/content/s2s_frontend.log`.